# RAG Assignment: Baseline vs. Re-Ranking

**Course:** CL3410 Language Models and Agents · Module 7 (RAG) · **Due:** 19 Oct 2026

This notebook implements two retrieval workflows over the **exact same chunked corpus** of a single PDF and compares them:

| | Pipeline 1 (baseline) | Pipeline 2 (re-ranking) |
|---|---|---|
| Retrieve | bi-encoder + FAISS inner product, **top 10** | the **same** top 10 from Pipeline 1 |
| Re-rank | none | cross-encoder scores all 10 (query, chunk) pairs, re-sorted descending |
| To the LLM | top 3 → **Answer 1** | top 3 → **Answer 2** (identical prompt structure) |

```
 OFFLINE:  PDF -> extract & clean -> structure-aware chunks (frozen, hashed) -> bi-encoder -> FAISS IndexFlatIP
 ONLINE :  query -> bi-encoder -> FAISS top-10 --+--> top-3 ---------------------------> prompt -> LLM -> Answer 1
                                                 |
                                                 +--> cross-encoder x 10 -> sort -> top-3 -> prompt -> LLM -> Answer 2
```

**How to run:** create the venv from `requirements.txt`, put keys in `.env` (see `.env.example`), then *Restart kernel and Run All*.
Sections are added stage by stage: setup → document → chunking → index → retrieval/re-ranking → generation → queries → results → failure analysis → latency & scaling → summary.

## 1. Setup

Imports, a single `CONFIG` dict (every tunable lives here), seeds, and the compute device.
Model weights are cached **inside the project** (`.cache/huggingface`, git-ignored) so nothing is written outside the repo folder.

In [1]:
import os, sys, time, json, hashlib, platform, random, statistics
from pathlib import Path

ROOT = Path.cwd()
# Keep Hugging Face downloads inside the project folder (must be set BEFORE importing transformers / sentence-transformers).
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # keep notebook outputs clean

import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import torch
import faiss
import pymupdf
import sentence_transformers
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")   # API keys are read from .env / environment only; they are never printed or committed.

# Every tunable parameter in one place. After the query set is frozen (Stage 6) none of these may change.
CONFIG = {
    "pdf_path": None,                    # set in Stage 1 once the document is chosen
    "bi_encoder": "sentence-transformers/all-MiniLM-L6-v2",
    "cross_encoder": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "llm_provider": None,                # gemini | anthropic | ollama  (decided in Stage 5)
    "llm_model": None,                   # exact pinned model id        (decided in Stage 5)
    "temperature": 0.0,
    "max_output_tokens": 400,
    "chunk_max_tokens": 220,             # measured with the bi-encoder tokenizer, heading included; hard cap 256
    "chunk_overlap_tokens": 40,
    "k_retrieve": 10,                    # Pipeline 1 retrieves 10 ...
    "k_final": 3,                        # ... and both pipelines send 3 to the LLM
    "latency_warmup": 3,
    "latency_repeats": 20,
    "seed": 42,
    "device": None,                      # filled by the device benchmark below, then fixed for all later timings
}

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])

for d in ["data", "queries", "artifacts/results", "artifacts/figures", "report"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

Project root: /Users/hritikranjan/Desktop/ Folder/SEM 3/LMA/A1_RAG/RAG-BASELINE-VS-RERANKING


### 1.1 Environment record

Latency numbers depend on hardware and library versions, so they are recorded once here.

In [2]:
import subprocess
def _sh(cmd):
    try: return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception: return "n/a"

ENV_INFO = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu": _sh("sysctl -n machdep.cpu.brand_string") if platform.system() == "Darwin" else platform.processor(),
    "ram_gb": round(int(_sh("sysctl -n hw.memsize") or 0) / 2**30, 1) if platform.system() == "Darwin" else "n/a",
    "torch": torch.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "faiss": faiss.__version__,
    "pymupdf": pymupdf.__version__,
    "mps_available": torch.backends.mps.is_available(),
    "cuda_available": torch.cuda.is_available(),
}
pd.Series(ENV_INFO, name="value").to_frame()

,value
python,3.12.13
platform,macOS-27.0-arm64-arm-64bit
cpu,Apple M4
ram_gb,24.0
torch,2.14.1
sentence_transformers,6.1.0
faiss,1.15.1
pymupdf,1.28.2
mps_available,True
cuda_available,False


### 1.2 Model smoke tests

Both models must load and behave as expected before anything is built on them.
- The bi-encoder **silently truncates input at `max_seq_length` tokens** (256 for all-MiniLM-L6-v2), which drives the chunk-size limit in Stage 2.
- The cross-encoder outputs a raw **logit** (any real number), not a probability.

In [3]:
t0 = time.perf_counter()
bi_encoder = SentenceTransformer(CONFIG["bi_encoder"], device="cpu")
t_bi_load = time.perf_counter() - t0

t0 = time.perf_counter()
cross_encoder = CrossEncoder(CONFIG["cross_encoder"], max_length=512, device="cpu")
t_ce_load = time.perf_counter() - t0

print(f"bi-encoder   : {CONFIG['bi_encoder']}  load={t_bi_load:.1f}s  dim={bi_encoder.get_embedding_dimension()}  max_seq_length={bi_encoder.max_seq_length}")
print(f"cross-encoder: {CONFIG['cross_encoder']}  load={t_ce_load:.1f}s")

assert bi_encoder.max_seq_length == 256, "unexpected bi-encoder max_seq_length: re-check the chunk-size cap"
assert bi_encoder.get_embedding_dimension() == 384

# bi-encoder: normalised vectors -> inner product == cosine
v = bi_encoder.encode(["a quick test sentence", "a quick test sentence"], normalize_embeddings=True)
print("bi-encoder embedding shape:", v.shape, "| norm:", float(np.linalg.norm(v[0])), "| self-cosine:", float(v[0] @ v[1]))

# cross-encoder: one raw logit per (query, passage) pair
pairs = [("what is the capital of france", "Paris is the capital and largest city of France."),
         ("what is the capital of france", "Bananas are rich in potassium.")]
scores = cross_encoder.predict(pairs)
print("cross-encoder logits (relevant, irrelevant):", [round(float(s), 3) for s in scores])
assert scores[0] > scores[1], "cross-encoder should score the relevant pair higher"

bi-encoder   : sentence-transformers/all-MiniLM-L6-v2  load=5.5s  dim=384  max_seq_length=256
cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2  load=5.7s
bi-encoder embedding shape: (2, 384) | norm: 1.0000001192092896 | self-cosine: 1.000000238418579
cross-encoder logits (relevant, irrelevant): [7.96, -11.127]


### 1.3 Device choice (benchmarked once, then fixed)

On Apple Silicon, MPS can be *slower* than CPU for tiny batches such as 10 pairs. The cross-encoder is timed on each available device **once**; the faster one is used for every later measurement so that the latency comparison is consistent. Warm-up runs are excluded.

In [4]:
def _bench_ce(device, n_warm=3, n_rep=20):
    ce = CrossEncoder(CONFIG["cross_encoder"], max_length=512, device=device)
    pairs10 = [("what happens if a player cannot pay the cost",
                "Players may not exceed the hand limit. If a player cannot pay a cost, the effect does not resolve. " * 3)] * 10
    for _ in range(n_warm): ce.predict(pairs10)
    ts = []
    for _ in range(n_rep):
        t0 = time.perf_counter(); ce.predict(pairs10); ts.append((time.perf_counter() - t0) * 1000)
    return statistics.median(ts)

candidates = ["cpu"] + (["mps"] if torch.backends.mps.is_available() else []) + (["cuda"] if torch.cuda.is_available() else [])
bench = {d: _bench_ce(d) for d in candidates}
CONFIG["device"] = min(bench, key=bench.get)
print("median ms for 10 (query, chunk) pairs per device:", {k: round(v, 1) for k, v in bench.items()})
print("DEVICE fixed for all later timings ->", CONFIG["device"])

# Reload both models on the chosen device; these are the objects used for the rest of the notebook.
bi_encoder = SentenceTransformer(CONFIG["bi_encoder"], device=CONFIG["device"])
cross_encoder = CrossEncoder(CONFIG["cross_encoder"], max_length=512, device=CONFIG["device"])

median ms for 10 (query, chunk) pairs per device: {'cpu': 20.1, 'mps': 14.8}
DEVICE fixed for all later timings -> mps


### 1.4 LLM connection

The LLM provider and model are chosen in **Stage 5** (Generation), once the retrieval side is working. Retrieval and re-ranking (Stages 2–4) need no LLM, so they can run without any API key.